# Project 2: Sales Prediction Using Python
**Oasis Infobyte Data Science Internship**

**Goal:** Predict how many units of a product will be sold based on how much money is spent on advertising (TV, Radio, Newspaper).

This is a **regression** problem (we predict a number, not a category).

**Steps:** Import libraries, load data, understand data, EDA, preprocessing, train models, evaluate, predict, conclusion.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
print("Libraries imported successfully!")

## 2. Load the Dataset
We use the well-known **Advertising dataset** (200 rows). Columns:
- `TV`, `Radio`, `Newspaper`: advertising budget (in thousands of dollars)
- `Sales`: units sold (in thousands)

The code below downloads it automatically from the internet. If both links ever fail, it will ask you to **upload `Advertising.csv`** yourself (you can download it from Kaggle: search "Advertising dataset" or "Sales Prediction Oasis Infobyte").

In [ ]:
urls = [
    "https://raw.githubusercontent.com/justmarkham/scikit-learn-videos/master/data/Advertising.csv",
    "https://www.statlearning.com/s/Advertising.csv",
]

df = None
for url in urls:
    try:
        df = pd.read_csv(url)
        print("Loaded from:", url)
        break
    except Exception as e:
        print("Could not load from", url)

# Fallback: manual upload in Google Colab
if df is None:
    from google.colab import files
    print("Please choose your Advertising.csv file...")
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))

# The original file has an extra index column; remove it if present
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df.columns = [c.strip() for c in df.columns]
df.head()

## 3. Understand the Data

In [ ]:
print("Shape (rows, columns):", df.shape)
df.info()

In [ ]:
df.describe()

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# 4.1 Distribution of each column
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), df.columns):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Relationship between each advertising channel and Sales
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["TV", "Radio", "Newspaper"]):
    sns.regplot(x=col, y="Sales", data=df, ax=ax,
                scatter_kws={"alpha": 0.6}, line_kws={"color": "red"})
    ax.set_title(f"{col} vs Sales")
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Correlation heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# 4.4 Pair plot
sns.pairplot(df)
plt.show()

In [ ]:
# 4.5 Box plots to look for outliers
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, df.columns):
    sns.boxplot(y=df[col], ax=ax, color="lightgreen")
    ax.set_title(col)
plt.tight_layout()
plt.show()

**EDA Observations**
- **TV** has the strongest positive relationship with Sales (correlation about 0.78).
- **Radio** has a moderate positive relationship (about 0.58).
- **Newspaper** has a weak relationship (about 0.23).
- There are no missing values and almost no outliers.

## 5. Preprocessing
The data is clean, so we only need to separate inputs (X) from the target (y) and split into train and test sets.

In [ ]:
X = df[["TV", "Radio", "Newspaper"]]   # inputs
y = df["Sales"]                        # what we want to predict

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Training rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])

## 6. Train Models
We train 5 regression models. We do not need scaling here because these models work fine on this data.

**Metrics explained:**
- **MAE**: average size of the error (lower is better)
- **RMSE**: like MAE but punishes big errors more (lower is better)
- **R² score**: how much of the variation in sales the model explains (closer to 1 is better)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2 Score": r2_score(y_test, pred),
    })

results = pd.DataFrame(rows).sort_values("R2 Score", ascending=False).reset_index(drop=True)
results.round(4)

In [ ]:
# Visual comparison of R2 scores
plt.figure(figsize=(8, 4))
sns.barplot(x="R2 Score", y="Model", data=results, palette="viridis")
plt.title("Model Comparison (higher R2 is better)")
plt.show()

## 7. Evaluate the Best Model

In [ ]:
# Pick the model with the highest R2 score automatically
best_name = results.loc[0, "Model"]
best_model = models[best_name]
print("Best model:", best_name)

y_pred = best_model.predict(X_test)
print("MAE  :", round(mean_absolute_error(y_test, y_pred), 4))
print("MSE  :", round(mean_squared_error(y_test, y_pred), 4))
print("RMSE :", round(np.sqrt(mean_squared_error(y_test, y_pred)), 4))
print("R2   :", round(r2_score(y_test, y_pred), 4))

In [ ]:
# Actual vs Predicted: points close to the red line = good predictions
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.7, color="teal")
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title(f"Actual vs Predicted ({best_name})")
plt.show()

In [ ]:
# Residuals = actual - predicted. They should be scattered around 0.
residuals = y_test - y_pred
plt.figure(figsize=(7, 4))
sns.histplot(residuals, kde=True, color="orange")
plt.title("Distribution of Errors (Residuals)")
plt.xlabel("Error")
plt.show()

In [ ]:
# Cross-validation for a more reliable score
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring="r2")
print("R2 for each fold:", np.round(cv_scores, 3))
print("Average R2      :", round(cv_scores.mean(), 4))

In [ ]:
# How much does each advertising channel matter?
lr = LinearRegression().fit(X_train, y_train)
coef = pd.DataFrame({"Feature": X.columns, "Coefficient": lr.coef_})
print(coef)
print("Intercept:", round(lr.intercept_, 4))
print("\nMeaning: spending 1 more unit on TV changes Sales by about", round(lr.coef_[0], 3), "units (all else equal).")

# Feature importance from Random Forest
rf = models["Random Forest"]
pd.Series(rf.feature_importances_, index=X.columns).sort_values().plot(
    kind="barh", color="purple", title="Random Forest Feature Importance", figsize=(6, 3))
plt.show()

## 8. Predict Sales for a New Advertising Budget

In [ ]:
new_data = pd.DataFrame({
    "TV": [150, 230, 50],
    "Radio": [30, 40, 10],
    "Newspaper": [20, 30, 10],
})
new_data["Predicted_Sales"] = best_model.predict(new_data[["TV", "Radio", "Newspaper"]]).round(2)
new_data

## 9. Conclusion
- The dataset is clean with no missing values.
- **TV advertising** has the biggest effect on sales, followed by **Radio**. **Newspaper** adds very little.
- Several models perform well (R² around 0.9 or higher); Linear Regression is simple and performs nearly as well as the complex models, while Random Forest / Gradient Boosting usually score highest.
- Business advice: spend more on TV and Radio ads, and reduce Newspaper spending.

**Limitations:** the dataset is small (200 rows) and only includes advertising spend. Real sales also depend on season, price, competition, etc.